In [ ]:
import os
import random
import numpy as np
import torch
from torch import nn, optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

from ssonn.metrics.edge_finder import EdgeFinder
from ssonn.metrics.nonlinearity_metrics import AbsGradientEdgeMetric
from ssonn.train.train import (
    convert_to_scaled_net, eval_one_epoch, expand_model,
    train_one_epoch, weights_init,
)

SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
g = torch.Generator().manual_seed(SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28):
        super().__init__()
        self.fc_output = nn.Linear(input_size, 10)

    def forward(self, x):
        return self.fc_output(x)

In [ ]:
hyperparams = {
    'epochs': 50,  # full-training epochs after architecture search
    'architecture_epochs': 16,
    'architecture_t_max': 4,
    'batch_size': 256,
    'learning_rate': 1e-3,
    'weight_decay': 0,
    'edge_importance_metric': AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    'edge_score_aggregation': 'mean',
    'expansion_thresholds': {'fc_output': 0.4},
    'max_new_edges_per_expansion': 2000,
}

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices)

train_loader = DataLoader(
    train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g
)
finder_loader = DataLoader(finder_dataset, batch_size=hyperparams['batch_size'])
val_loader = DataLoader(val_dataset, batch_size=hyperparams['batch_size'])

In [ ]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, [model.fc_output], device)

In [ ]:
from comet_ml import start

experiment = start(
    api_key=os.environ.get('COMET_API_KEY'),
    workspace=os.environ.get('COMET_WORKSPACE'),
    project_name='mnist',
)
experiment.log_parameters(hyperparams)
experiment.set_name('mnist architecture first')

In [ ]:
criterion = nn.CrossEntropyLoss()

def make_optimizer(model):
    return optim.SGD(
        model.parameters(),
        lr=hyperparams['learning_rate'],
        weight_decay=hyperparams['weight_decay'],
        momentum=0.9,
    )

def fix_sparsity(model):
    # Expansion replaces the main weight parameter. Refresh its mask and hook
    # without touching the learned values.
    for module in model.modules():
        if not hasattr(module, 'weight') or module.weight is None:
            continue
        mask = (module.weight.detach() != 0).to(module.weight.dtype)
        if hasattr(module, 'sparsity_mask'):
            module.sparsity_mask = mask
        else:
            module.register_buffer('sparsity_mask', mask)
        if not getattr(module.weight, '_has_sparsity_hook', False):
            module.weight.register_hook(lambda grad, fixed_mask=mask: grad * fixed_mask)
            module.weight._has_sparsity_hook = True

def train_architecture_first(model):
    training_epochs = hyperparams['epochs']
    architecture_epochs = hyperparams['architecture_epochs']
    architecture_t_max = hyperparams['architecture_t_max']
    total_epochs = architecture_epochs + training_epochs
    assert architecture_epochs > 0 and training_epochs > 0

    model = model.to(device)
    optimizer = make_optimizer(model)
    scheduler = CosineAnnealingLR(optimizer, T_max=architecture_t_max)
    edge_finder = EdgeFinder(
        metric=hyperparams['edge_importance_metric'],
        dataloader=finder_loader,
        device=device,
        aggregation_mode=hyperparams['edge_score_aggregation'],
        max_to_replace=hyperparams['max_new_edges_per_expansion'],
    )

    for epoch in range(total_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = eval_one_epoch(
            model, criterion, val_loader, 'classification', device
        )

        epoch_number = epoch + 1
        architecture_phase = epoch_number <= architecture_epochs
        expansion_epoch = (
            epoch_number % architecture_t_max == 0
            or epoch_number == architecture_epochs
        )

        if architecture_phase and expansion_epoch:
            expanded, new_edges = expand_model(
                model, edge_finder, hyperparams['expansion_thresholds']
            )
            if expanded:
                fix_sparsity(model)
            print(f'Expansion: {new_edges} new edges')
            experiment.log_metric('expansion event', new_edges, step=epoch_number)

            if epoch_number < architecture_epochs:
                optimizer = make_optimizer(model)
                scheduler = CosineAnnealingLR(optimizer, T_max=architecture_t_max)

        if epoch_number == architecture_epochs:
            # The architecture is ready. Initialize it once and train all weights.
            model.apply(weights_init)
            fix_sparsity(model)
            optimizer = make_optimizer(model)
            scheduler = CosineAnnealingLR(
                optimizer, T_max=training_epochs
            )
            print('Architecture fixed; full training started')
            experiment.log_metric('architecture ready', 1, step=epoch_number)
        elif not (architecture_phase and expansion_epoch):
            scheduler.step()

        phase = 'architecture' if architecture_phase else 'training'
        non_zero = sum((p != 0).sum().item() for p in model.parameters())
        total = sum(p.numel() for p in model.parameters())
        experiment.log_metrics({
            'train loss': train_loss,
            'train accuracy': train_acc,
            'val loss': val_loss,
            'val accuracy': val_acc,
            'lr': optimizer.param_groups[0]['lr'],
            'total params': total,
            'params amount': non_zero,
            'architecture phase': int(architecture_phase),
        }, step=epoch_number)
        print(
            f'Epoch {epoch_number:02d}/{total_epochs} [{phase}] | '
            f'T-Loss: {train_loss:.4f} | V-Loss: {val_loss:.4f} | '
            f'Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]["lr"]:.1e} | '
            f'Params: {non_zero}/{total}'
        )

    return model

trained_model = train_architecture_first(model)

In [ ]:
experiment.end()

In [ ]:
torch.save(trained_model, 'mnist_architecture_first.pt')